# Tutorial 3 – Forced vibration of a 1-DoF system
**Vibrations – Centrale Nantes**

Animations of the two forced systems of Tutorial 3:
* **Animation 1** – mass–spring–damper loaded by a harmonic force (group A),
* **Animation 2** – pendulum in air loaded by a harmonic torque (group B),
* **Animation 3** – the same system forced below, at and above its natural frequency.

**How to use it:** run all the cells (*Run → Run all cells*). In the interactive figures, move the **sliders** to change the parameters and use the **Play/Pause** buttons below the figure.
The code cells are hidden (click on them to show the code).

In [ ]:
#@title Setup (run first): interactive figures with sliders
import sys, subprocess
if "pyodide" in sys.modules:                     # JupyterLite (in the browser)
    import piplite
    await piplite.install(["ipympl", "ipywidgets"])
elif "google.colab" in sys.modules:              # Google Colab
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ipympl"])
    from google.colab import output
    output.enable_custom_widget_manager()
try:
    import ipympl
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    print("ipympl is not installed: the animations will be shown without sliders (pip install ipympl).")

In [ ]:
#@title Helper functions (run this cell first)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

plt.rcParams.update({"figure.dpi": 90, "animation.embed_limit": 60, "font.size": 10,
                     # LaTeX-like rendering without a LaTeX installation: $...$ formulas and text in STIX (Times-like)
                     "mathtext.fontset": "stix", "font.family": "STIXGeneral"})
BLUE, RED, GREEN, GREY = "#1f4e99", "#a3201d", "#2e6b30", "#777777"


def rk4(f, y0, t):
    """Fixed-step Runge-Kutta 4 integration of dy/dt = f(t, y)."""
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for i in range(len(t) - 1):
        h = t[i + 1] - t[i]
        k1 = f(t[i], y[i])
        k2 = f(t[i] + h / 2, y[i] + h / 2 * k1)
        k3 = f(t[i] + h / 2, y[i] + h / 2 * k2)
        k4 = f(t[i] + h, y[i] + h * k3)
        y[i + 1] = y[i] + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return y


def spring_xy(p0, p1, n=7, width=0.012, lead=0.15):
    """Zig-zag polyline of a spring between points p0 and p1."""
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    L = np.linalg.norm(p1 - p0)
    u = (p1 - p0) / L
    v = np.array([-u[1], u[0]])
    pts = [p0, p0 + lead * L * u]
    for i in range(2 * n):
        s = lead * L + (1 - 2 * lead) * L * (i + 0.5) / (2 * n)
        pts.append(p0 + s * u + (width if i % 2 == 0 else -width) * v)
    pts += [p1 - lead * L * u, p1]
    pts = np.array(pts)
    return pts[:, 0], pts[:, 1]


def hatch(ax, x0, x1, y, down=True, n=12, size=0.012):
    """Draw a fixed support (line + hatching)."""
    ax.plot([x0, x1], [y, y], color="k", lw=2)
    s = -size if down else size
    for x in np.linspace(x0, x1, n):
        ax.plot([x, x + size], [y, y - s], color="k", lw=0.8)


def make_anim(fig, update, nframes, fps=25):
    """Build an animation and remember what is needed to export it."""
    anim = animation.FuncAnimation(fig, update, frames=nframes, interval=1000 / fps, blit=False)
    anim.update_func, anim.nframes = update, nframes
    return anim


def show(anim):
    """Display an animation in the notebook (works in Jupyter, JupyterLite and Colab)."""
    plt.close(anim._fig)
    display(HTML(anim.to_jshtml()))


def export_frames(anim, name, folder="gifs", dpi=120):
    """Teacher only: save the frames as name-0.png, name-1.png, ... for the LaTeX animate package."""
    import os
    os.makedirs(f"{folder}/{name}", exist_ok=True)
    for i in range(anim.nframes):
        anim.update_func(i)
        anim._fig.savefig(f"{folder}/{name}/{name}-{i}.png", dpi=dpi)
    print(f"{anim.nframes} frames saved in {folder}/{name}/ (last index {anim.nframes - 1})")


# ---------------------------------------------------------------------------
# Live animation: sliders drawn on the figure + Play/Pause widget below it.
# Needs ipympl (Jupyter, JupyterLite, VS Code, Colab). The time steps are driven
# by the ipywidgets "Play" widget (timer in the browser), which also works in
# JupyterLite where matplotlib timers do not run.
# Without ipympl, the animation is shown with the default parameters.
# ---------------------------------------------------------------------------
from matplotlib.widgets import Slider

INTERACTIVE = "ipympl" in plt.get_backend().lower() or "widget" in plt.get_backend().lower()


class LiveAnimation:
    """
    sliders : list of (name, label, vmin, vmax, vinit, step)
    compute : function(params dict) -> data dict (solution on a fine time grid)
    setup   : function(data) -> None, updates the static elements (curves, limits...)
    frame   : function(data, t) -> None, updates the moving elements at time t
    T       : simulated duration (s), played in a loop
              (compute may return data["T"] to change it, e.g. a whole number of periods)
    slow    : slow motion factor
    """

    def __init__(self, fig, sliders, compute, setup, frame, T, slow=1, fps=20):
        self.fig, self.compute, self.setup, self.frame = fig, compute, setup, frame
        self.T, self.slow, self.fps = T, slow, fps
        n = len(sliders)
        fig.subplots_adjust(bottom=0.08 + 0.045 * n)
        self.sliders = {}
        for j, (name, label, vmin, vmax, vinit, step) in enumerate(sliders):
            ax = fig.add_axes([0.22, 0.02 + 0.045 * (n - 1 - j), 0.5, 0.025])
            s = Slider(ax, label, vmin, vmax, valinit=vinit, valstep=step)
            s.on_changed(self._changed)
            self.sliders[name] = s
        for a in fig.axes:                        # drawings with equal aspect: keep them aligned at the top
            if a.get_aspect() == 1.0:
                a.set_anchor("N")
        self.i = 0
        self.data = self.compute(self.params())
        self._loop()
        self.setup(self.data)
        self.frame(self.data, 0.0)
        if INTERACTIVE:
            import ipywidgets as W
            fig.canvas.header_visible = False
            # Play button (timer in the browser) linked to a time slider: the slider can also be moved by hand
            self.play = W.Play(value=0, min=0, max=self.nframes - 1, step=1,
                               interval=int(1000 / fps), repeat=True, description="Play")
            self.tslider = W.IntSlider(value=0, min=0, max=self.nframes - 1, readout=False,
                                       layout=W.Layout(width="45%"))
            W.jslink((self.play, "value"), (self.tslider, "value"))
            self.tlabel = W.Label()
            self.out = W.Output()                 # error messages of the callbacks (if any)
            self.tslider.observe(self._tick, names="value")
            plt.show()                            # figure first, controls below
            display(W.VBox([W.HBox([self.play, self.tslider, self.tlabel]), self.out]))   # paused by default
            self._show_t()
        else:
            print("Static version (default parameters): install ipympl to use the sliders.")
            show(make_anim(fig, lambda i: self.frame(self.data, i / self.nframes * self.Tloop), self.nframes, fps))

    def params(self):
        return {name: s.val for name, s in self.sliders.items()}

    def _loop(self):
        """Loop duration (from compute if given) and number of frames of one loop."""
        self.Tloop = self.data.get("T", self.T)
        self.nframes = max(2, int(round(self.Tloop * self.fps * self.slow)))

    def _t(self):
        return (self.i % self.nframes) / self.nframes * self.Tloop

    def _show_t(self):
        self.tlabel.value = f"t = {self._t():.2f} s"

    def _report(self, err):
        import traceback
        with self.out:
            traceback.print_exc()

    def _changed(self, _):
        try:
            self.data = self.compute(self.params())
            self._loop()
            if hasattr(self, "play"):
                self.i = min(self.i, self.nframes - 1)
                self.play.max = self.tslider.max = self.nframes - 1
                self._show_t()
            self.setup(self.data)
            self.frame(self.data, self._t())
            self.fig.canvas.draw_idle()
        except Exception as err:
            self._report(err)

    def _tick(self, change):
        try:
            self.i = change["new"]
            self.frame(self.data, self._t())
            self._show_t()
            self.fig.canvas.draw_idle()
        except Exception as err:
            self._report(err)

    def export(self, name, folder="gifs", dpi=120):
        """Teacher only: frames for the LaTeX animate package (current slider values)."""
        export_frames(make_anim(self.fig, lambda i: self.frame(self.data, i / self.nframes * self.Tloop),
                                self.nframes, self.fps), name, folder, dpi)


def period(t, q):
    """Period measured from the upward zero crossings of q(t)."""
    idx = np.where((q[:-1] < 0) & (q[1:] >= 0))[0]
    return np.mean(np.diff(t[idx])) if len(idx) > 1 else np.nan


class Dashpot:
    """Viscous damper drawn between p0 (cylinder side) and p1 (piston side).
    The geometry is fixed by the first call to set() (or after reset()): the piston then moves inside the cylinder."""

    def __init__(self, ax, width=0.01, **kw):
        kw.setdefault("color", "k"); kw.setdefault("lw", 1.2)
        self.w, self.geom = width, None
        self.lines = [ax.plot([], [], **kw)[0] for _ in range(3)]

    def reset(self):
        self.geom = None

    def set(self, p0, p1):
        p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
        L = np.linalg.norm(p1 - p0); u = (p1 - p0) / L; v = np.array([-u[1], u[0]])
        if self.geom is None:
            self.geom = (0.1 * L, 0.65 * L, 0.6 * L)          # stem, cylinder length, piston rod length
        a, cyl, rod = self.geom
        P = lambda s, d=0.0: p0 + s * u + d * v
        stem = np.array([p0, P(a)])
        cylinder = np.array([P(a + cyl, self.w), P(a, self.w), P(a, -self.w), P(a + cyl, -self.w)])
        sp = L - rod
        piston = np.array([P(sp, 0.8 * self.w), P(sp, -0.8 * self.w), P(sp), p1])
        for line, pts in zip(self.lines, [stem, cylinder, piston]):
            line.set_data(pts[:, 0], pts[:, 1])


def double_pendulum_rhs(g, L):
    """Exact equations of a double pendulum with two equal masses and two equal lengths (absolute angles)."""
    def f(_, s):
        t1, t2, w1, w2 = s
        d = t1 - t2
        den = L * (3 - np.cos(2 * d))
        a1 = (-3 * g * np.sin(t1) - g * np.sin(t1 - 2 * t2) - 2 * np.sin(d) * (w2 ** 2 * L + w1 ** 2 * L * np.cos(d))) / den
        a2 = (2 * np.sin(d) * (2 * w1 ** 2 * L + 2 * g * np.cos(t1) + w2 ** 2 * L * np.cos(d))) / den
        return np.array([w1, w2, a1, a2])
    return f


def draw_two_masses(ax, x1, x2, objs=None, gap=0.3, w=0.12, h=0.1, scale=1.0):
    """Two masses between two walls (walls at 0 and 3*gap+2*w); x1, x2 in the units of the drawing."""
    xa = gap + w / 2 + scale * x1
    xb = 2 * gap + 1.5 * w + scale * x2
    xr = 3 * gap + 2 * w
    if objs is None:
        hatch_v = lambda x, s: [ax.plot([x, x], [-0.02, h + 0.06], "k", lw=2)] + \
            [ax.plot([x, x + s], [y, y + 0.02], "k", lw=0.7) for y in np.linspace(-0.02, h + 0.04, 7)]
        hatch_v(0, -0.025); hatch_v(xr, 0.025)
        ax.plot([0, xr], [-0.02, -0.02], "k", lw=1)
        sps = [ax.plot([], [], "k", lw=1.1)[0] for _ in range(3)]
        boxes = [plt.Rectangle((0, 0), w, h, fc="#f4c7c3", ec="k") for _ in range(2)]
        for b in boxes:
            ax.add_patch(b)
        objs = (sps, boxes)
    sps, boxes = objs
    yc = h / 2
    ends = [(0, xa - w / 2), (xa + w / 2, xb - w / 2), (xb + w / 2, xr)]
    for s, (p, q) in zip(sps, ends):
        s.set_data(*spring_xy((p, yc), (q, yc), n=6, width=0.025))
    boxes[0].set_x(xa - w / 2); boxes[1].set_x(xb - w / 2)
    return objs


## Animation 1 – Forced mass–spring–damper (group A)
The system starts **at rest** and is loaded by $F(t) = F_0\cos\Omega t$. The force is drawn as an arrow, the right-hand graphs show the response $u(t)$ split into the **free solution** (transient, $u_h$) and the **forced solution** (steady state, $u_p$), then the steady-state amplitude $U$ and phase lag $\varphi$ of the displacement versus the forcing frequency (the red dot is the current $\Omega$).

**Observe:** when the free solution has died out (end of the transient); how the amplitude and the phase change when $\Omega$ gets close to $\sqrt{k/m}$; the effect of $c$ on the duration of the transient.

In [ ]:
#@title Animation 1
def animation_1():
    m, k, g, Tsim = 0.5, 200.0, 9.81, 2.0
    L0, hbox = 0.06, 0.02
    w0 = np.sqrt(k / m)

    def amp(W, c, F0):
        return F0 / np.sqrt((k - m * W ** 2) ** 2 + (c * W) ** 2)

    def phase(W, c):                                     # phase lag of u behind F (degrees, 0 to 180)
        return np.degrees(np.arctan2(c * W, k - m * W ** 2))

    def compute(p):
        W, c, F0 = p["W"], p["c"], p["F0"]
        tf = np.linspace(0, Tsim, 4001)
        u = rk4(lambda t, s: np.array([s[1], (F0 * np.cos(W * t) - c * s[1] - k * s[0]) / m]), [0.0, 0.0], tf)[:, 0]
        up = amp(W, c, F0) * np.cos(W * tf - np.radians(phase(W, c)))     # forced solution (steady state)
        return dict(tf=tf, u=u, up=up, W=W, c=c, F0=F0)

    fig = plt.figure(figsize=(9.5, 8))
    gs = fig.add_gridspec(3, 2, width_ratios=[1, 2], top=0.95, wspace=0.28, hspace=0.6)
    ax = fig.add_subplot(gs[:, 0]); axq = fig.add_subplot(gs[0, 1]); axf = fig.add_subplot(gs[1, 1]); axp = fig.add_subplot(gs[2, 1])
    ax.set_aspect("equal"); ax.axis("off")
    ye = m * g / k
    ax.set_xlim(-0.07, 0.07); ax.set_ylim(-(L0 + ye + 0.025 + hbox + 0.005), 0.01)
    hatch(ax, -0.05, 0.05, 0.0, down=False, size=0.01)
    ax.axhline(-(L0 + ye), color=GREY, ls="--", lw=0.8)
    sp, = ax.plot([], [], "k", lw=1.2); dp = Dashpot(ax, width=0.008)
    box = plt.Rectangle((-0.03, 0), 0.06, hbox, fc="#f4c7c3", ec="k"); ax.add_patch(box)
    arrow = ax.annotate("", xy=(0, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=BLUE, lw=2))
    ax.set_title("forced mass–spring–damper")
    curve, = axq.plot([], [], color=BLUE, lw=1.8, label=r"total $u$")
    cfree, = axq.plot([], [], color=GREEN, lw=1.2, ls=":", label=r"free $u_h$ (transient)")
    cforc, = axq.plot([], [], color="k", lw=1, ls="--", label=r"forced $u_p$ (steady state)")
    axq.legend(fontsize=7, loc="upper right", ncol=3)
    cursor = axq.axvline(0, color="k", lw=0.8)
    axq.set_xlim(0, Tsim); axq.set_xlabel(r"$t$ (s)"); axq.set_ylabel(r"$u$ (cm)"); axq.grid(alpha=0.3)
    Ws = np.linspace(0.01, 3 * w0, 600)
    frf, = axf.plot([], [], color=BLUE, lw=2)
    dot, = axf.plot([], [], "o", color=RED)
    axf.set_xlim(0, 3 * w0); axf.set_yscale("log"); axf.set_ylim(1e-2, 1e2); axf.set_ylabel(r"$U$ (cm)")
    axf.grid(alpha=0.3, which="both")
    ph, = axp.plot([], [], color=BLUE, lw=2)
    pdot, = axp.plot([], [], "o", color=RED)
    for a_ in (axf, axp):
        a_.axvline(w0, color=GREY, ls=":", lw=1)
    axp.text(w0, 150, r" $\omega_0$", color=GREY, fontsize=10, va="center")
    axp.set_xlim(0, 3 * w0); axp.set_ylim(0, 180); axp.set_yticks([0, 90, 180])
    axp.set_xlabel(r"$\Omega$ (rad/s)"); axp.set_ylabel(r"$\varphi$ (°)"); axp.grid(alpha=0.3)

    def setup(d):
        curve.set_data(d["tf"], 100 * d["u"]); cforc.set_data(d["tf"], 100 * d["up"]); cfree.set_data(d["tf"], 100 * (d["u"] - d["up"]))
        a = max(np.max(np.abs(d["u"])), np.max(np.abs(d["u"] - d["up"])), 1e-4); axq.set_ylim(-110 * a, 140 * a)
        A = 100 * amp(Ws, d["c"], d["F0"]); frf.set_data(Ws, A); axf.set_ylim(A.min() / 2, A.max() * 2)
        dot.set_data([d["W"]], [100 * amp(d["W"], d["c"], d["F0"])])
        ph.set_data(Ws, phase(Ws, d["c"])); pdot.set_data([d["W"]], [phase(d["W"], d["c"])])
        axp.set_title(rf"steady state: $u = U\cos(\Omega t - \varphi)$,   $\varphi$ = {phase(d['W'], d['c']):.0f}°", fontsize=10)
        dp.reset()

    def frame(d, t):
        u = np.interp(t, d["tf"], d["u"]); amax = max(np.max(np.abs(d["u"])), 1e-4)
        y = ye + 0.02 * u / amax                         # displacement scaled to stay inside the drawing
        sp.set_data(*spring_xy((-0.015, 0), (-0.015, -(L0 + y)), width=0.007)); dp.set((0.015, 0), (0.015, -(L0 + y)))
        yb = -(L0 + y) - hbox / 2; f = np.cos(d["W"] * t)
        box.set_y(-(L0 + y) - hbox)
        arrow.set_position((0.045, yb)); arrow.xy = (0.045, yb - 0.03 * f)
        cursor.set_xdata([t, t])

    live = LiveAnimation(fig, [("W", r"$\Omega$ (rad/s)", 2, 60, 15, 0.5), ("c", r"$c$ (N s/m)", 0.2, 20.0, 2.0, 0.2),
                               ("F0", r"$F_0$ (N)", 0.1, 5.0, 2.0, 0.1)],
                         compute, setup, frame, Tsim, slow=2)
    fig.text(0.02, 0.005, r"(drawing: displacement scaled to the maximum of $u(t)$)", fontsize=7, color=GREY)
    return live

live1 = animation_1()

## Animation 2 – Forced pendulum in air (group B)
The pendulum starts **at rest** and is loaded by a torque $M(t) = M_0\cos\Omega t$ about $O$ (exact nonlinear motion), drawn as the equivalent force $M(t)/L$ on the mass. The response $\theta(t)$ is split into the forced solution $\theta_p$ of the linearized model (steady state) and the free solution $\theta_h = \theta - \theta_p$ (transient). The two lower graphs show the steady-state amplitude $\Theta$ and phase lag $\varphi$ of the **linearized** model versus the forcing frequency (the red dot is the current $\Omega$).

**Observe:** when the free solution has died out; the amplitude and the phase when $\Omega$ is close to $\sqrt{g/L}$. Is the small-angle assumption still valid?

In [ ]:
#@title Animation 2
def animation_2():
    m, L, g, Tsim = 0.2, 0.25, 9.81, 30.0
    w0 = np.sqrt(g / L)

    def amp(W, c, M0):                                   # linearized model, degrees
        return np.degrees(M0 / np.sqrt((m * g * L - m * L ** 2 * W ** 2) ** 2 + (c * L ** 2 * W) ** 2))

    def phase(W, c):
        return np.degrees(np.arctan2(c * L ** 2 * W, m * g * L - m * L ** 2 * W ** 2))

    def compute(p):
        W, c, M0 = p["W"], p["c"], p["M0"] / 1000
        tf = np.linspace(0, Tsim, 12001)
        th = rk4(lambda t, s: np.array([s[1], (M0 * np.cos(W * t) - c * L ** 2 * s[1] - m * g * L * np.sin(s[0])) / (m * L ** 2)]),
                 [0.0, 0.0], tf)[:, 0]
        thp = np.radians(amp(W, c, M0)) * np.cos(W * tf - np.radians(phase(W, c)))   # forced solution (linearized)
        return dict(tf=tf, th=th, thp=thp, W=W, c=c, M0=M0)

    fig = plt.figure(figsize=(9.5, 8))
    gs = fig.add_gridspec(3, 2, width_ratios=[1.2, 2], top=0.95, wspace=0.25, hspace=0.6)
    ax = fig.add_subplot(gs[:, 0]); axq = fig.add_subplot(gs[0, 1]); axf = fig.add_subplot(gs[1, 1]); axp = fig.add_subplot(gs[2, 1])
    ax.set_aspect("equal"); ax.axis("off"); R = 1.15 * L
    ax.set_xlim(-R, R); ax.set_ylim(-R, 0.25 * R)
    hatch(ax, -0.15 * R, 0.15 * R, 0.0, down=False, size=0.04 * R)
    ax.plot([0, 0], [0, -L], color=GREY, ls="--", lw=0.8)
    rod, = ax.plot([], [], "k", lw=1.5); bob, = ax.plot([], [], "o", color=RED, ms=12)
    tq = ax.annotate("", xy=(0, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=BLUE, lw=2))
    ax.text(0.02, 0.02, r"arrow: $M(t)/L$", transform=ax.transAxes, color=BLUE, fontsize=9)
    ax.set_title("forced pendulum in air")
    curve, = axq.plot([], [], color=RED, lw=1.4, label=r"total $\theta$")
    cfree, = axq.plot([], [], color=GREEN, lw=1.1, ls=":", label=r"free $\theta_h$")
    cforc, = axq.plot([], [], color="k", lw=0.9, ls="--", label=r"forced $\theta_p$ (linear)")
    axq.legend(fontsize=7, loc="upper right", ncol=3)
    cursor = axq.axvline(0, color="k", lw=0.8)
    axq.set_xlim(0, Tsim); axq.set_xlabel(r"$t$ (s)"); axq.set_ylabel(r"$\theta$ (°)"); axq.grid(alpha=0.3)
    info = axq.set_title("")
    Ws = np.linspace(0.01, 3 * w0, 600)
    frf, = axf.plot([], [], color=RED, lw=2); dot, = axf.plot([], [], "o", color=RED, mec="k")
    ph, = axp.plot([], [], color=RED, lw=2); pdot, = axp.plot([], [], "o", color=RED, mec="k")
    for a_ in (axf, axp):
        a_.axvline(w0, color=GREY, ls=":", lw=1); a_.set_xlim(0, 3 * w0); a_.grid(alpha=0.3, which="both")
    axp.text(w0, 150, r" $\omega_0$", color=GREY, fontsize=10, va="center")
    axf.set_yscale("log"); axf.set_ylabel(r"$\Theta$ (°, linear)")
    axp.set_ylim(0, 180); axp.set_yticks([0, 90, 180]); axp.set_xlabel(r"$\Omega$ (rad/s)"); axp.set_ylabel(r"$\varphi$ (°)")

    def setup(d):
        curve.set_data(d["tf"], np.degrees(d["th"])); cforc.set_data(d["tf"], np.degrees(d["thp"])); cfree.set_data(d["tf"], np.degrees(d["th"] - d["thp"]))
        a = max(np.degrees(np.max(np.abs(d["th"]))), 0.5)
        b = max(a, np.degrees(np.max(np.abs(d["th"] - d["thp"])))); axq.set_ylim(-1.2 * b, 1.55 * b)
        info.set_text(rf"maximum angle: {a:.1f}°   ($\Omega/\omega_0$ = {d['W'] / w0:.2f})")
        A = amp(Ws, d["c"], d["M0"]); frf.set_data(Ws, A); axf.set_ylim(A.min() / 2, A.max() * 2)
        dot.set_data([d["W"]], [amp(d["W"], d["c"], d["M0"])])
        ph.set_data(Ws, phase(Ws, d["c"])); pdot.set_data([d["W"]], [phase(d["W"], d["c"])])
        axp.set_title(rf"steady state (linear): $\theta = \Theta\cos(\Omega t - \varphi)$,   $\varphi$ = {phase(d['W'], d['c']):.0f}°", fontsize=10)

    def frame(d, t):
        th = np.interp(t, d["tf"], d["th"])
        x, y = L * np.sin(th), -L * np.cos(th)
        rod.set_data([0, x], [0, y]); bob.set_data([x], [y])
        f = 0.4 * L * np.cos(d["W"] * t)                 # equivalent force M(t)/L on the mass, perpendicular to the rod
        tq.set_position((x, y)); tq.xy = (x + f * np.cos(th), y + f * np.sin(th))
        cursor.set_xdata([t, t])

    live = LiveAnimation(fig, [("W", r"$\Omega$ (rad/s)", 1, 15, 6.0, 0.1), ("c", r"$c$ (N s/m)", 0.01, 0.5, 0.05, 0.01),
                               ("M0", r"$M_0$ (mN m)", 0.5, 20, 5, 0.5)],
                         compute, setup, frame, Tsim, slow=0.25)
    return live

live2 = animation_2()

## Animation 3 – Below, at and above the natural frequency
Three identical mass–spring–damper systems ($m = 0.5$ kg, $k = 200$ N/m, $c = 2$ N s/m) in **steady state**, forced at $\Omega = \omega_0/2$, $\Omega = \omega_0$ and $\Omega = 2\omega_0$. The blue arrow is the force, the curves show the force and the displacement (both normalized). The lower graphs locate the three cases on the amplitude and phase curves versus $\Omega/\omega_0$ ($\zeta = 0.1$).

**Observe:** the amplitude of each system and the delay between the force and the displacement: in phase, quarter period, phase opposition.

In [ ]:
#@title Animation 3
def animation_3():
    m, k, c, fps = 0.5, 200.0, 2.0, 25
    w0 = np.sqrt(k / m)
    etas, cols, mks = [0.5, 1.0, 2.0], [BLUE, RED, GREEN], ["o", "s", "^"]
    zeta = c / (2 * np.sqrt(k * m))
    def Hn(e):                                                                     # normalized complex amplitude U/u_st
        return 1 / (1 - e ** 2 + 2j * zeta * e)
    H = [1 / (k - m * (e * w0) ** 2 + 1j * c * e * w0) for e in etas]           # steady state: u = Re(H F0 e^{jWt})
    Tp = 2 * np.pi / (0.5 * w0) * 2                                                # two periods of the slowest system
    nf = 120
    t = np.linspace(0, Tp, nf)
    L0, hbox, ys = 0.06, 0.02, 0.015

    fig = plt.figure(figsize=(9.5, 8.2))
    gs = fig.add_gridspec(3, 6, height_ratios=[1.0, 0.8, 1.0], hspace=0.4, wspace=1.2, top=0.93, bottom=0.07)
    objs, curs = [], []
    for j, (e, h) in enumerate(zip(etas, H)):
        ax = fig.add_subplot(gs[0, 2 * j:2 * j + 2]); ax.set_aspect("equal"); ax.axis("off")
        ax.set_xlim(-0.06, 0.06); ax.set_ylim(-(L0 + ys + hbox + 0.015), 0.01)
        hatch(ax, -0.04, 0.04, 0.0, down=False, size=0.008)
        ax.axhline(-L0, color=GREY, ls="--", lw=0.8)
        ax.set_title(f"$\\Omega = {e:g}\\,\\omega_0$\n$U$ = {abs(h) * k:.2f} $u_{{st}}$,  $\\varphi$ = {-np.degrees(np.angle(h)):.0f}°", fontsize=9, color=cols[j])
        s, = ax.plot([], [], "k", lw=1.1); d = Dashpot(ax, width=0.007)
        b = plt.Rectangle((-0.025, 0), 0.05, hbox, fc="#f4c7c3", ec="k"); ax.add_patch(b)
        ar = ax.annotate("", xy=(0, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=BLUE, lw=2))
        objs.append((s, d, b, ar))
        axt = fig.add_subplot(gs[1, 2 * j:2 * j + 2])
        W = e * w0; tt = np.linspace(0, Tp, 600)
        axt.plot(tt, np.cos(W * tt), color=BLUE, lw=1, ls="--", label="force")
        axt.plot(tt, np.real(h * np.exp(1j * W * tt)) / abs(h), color=cols[j], lw=2, label="displacement")
        axt.set_ylim(-1.3, 1.3); axt.set_yticks([]); axt.set_xlabel(r"$t$ (s)", fontsize=8); axt.tick_params(labelsize=7)
        curs.append(axt.axvline(0, color="k", lw=0.8))
        if j == 0:
            axt.legend(loc="lower left", fontsize=7)
    umax = max(abs(h) for h in H)
    axA, axP = fig.add_subplot(gs[2, 0:3]), fig.add_subplot(gs[2, 3:6])
    es = np.linspace(0, 3, 600)
    axA.plot(es, np.abs(Hn(es)), "k", lw=1.5); axP.plot(es, -np.degrees(np.angle(Hn(es))), "k", lw=1.5)
    for e, col, mk in zip(etas, cols, mks):
        lab = rf"$\Omega = {e:g}\,\omega_0$"
        axA.plot([e], [abs(Hn(e))], mk, color=col, ms=9, mec="k", label=lab)
        axP.plot([e], [-np.degrees(np.angle(Hn(e)))], mk, color=col, ms=9, mec="k")
    axA.set_ylim(0, 1.15 / (2 * zeta)); axA.set_ylabel(r"$U / u_{st}$"); axA.legend(fontsize=7, loc="upper right")
    axP.set_ylim(0, 180); axP.set_yticks([0, 90, 180]); axP.set_ylabel(r"$\varphi$ (°)")
    for a_ in (axA, axP):
        a_.set_xlim(0, 3); a_.set_xlabel(r"$\Omega / \omega_0$"); a_.grid(alpha=0.3); a_.tick_params(labelsize=8)
    axA.set_title("amplitude", fontsize=9); axP.set_title(r"phase lag of $u$ behind $F$", fontsize=9)

    def update(i):
        for (s, d, b, ar), e, h, cur in zip(objs, etas, H, curs):
            W = e * w0
            u = np.real(h * np.exp(1j * W * t[i])) / umax * ys      # drawing scaled to the largest amplitude
            s.set_data(*spring_xy((-0.012, 0), (-0.012, -(L0 + u)), width=0.006)); d.set((0.012, 0), (0.012, -(L0 + u)))
            b.set_y(-(L0 + u) - hbox)
            yb = -(L0 + u) - hbox / 2; f = np.cos(W * t[i])
            ar.set_position((0.038, yb)); ar.xy = (0.038, yb - 0.025 * f)
            cur.set_xdata([t[i], t[i]])

    anim = make_anim(fig, update, nf, fps)
    show(anim)
    return anim

anim3 = animation_3()

---
*Teacher only – export the frames of an animation for the LaTeX `animate` package* (uncomment and run):

In [ ]:
# live1.export("td3MassSpringForced")
# live2.export("td3PendulumForced")
# export_frames(anim3, "td3BelowAtAbove")